In [9]:
import numpy as np
import sys
from kaggle_environments import evaluate, make, utils, agent
from kaggle_environments import make, evaluate
import random
import inspect
import base64
import math

import torch
import torch.nn as nn
import torch.optim as optim

In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [11]:
env = make('connectx', debug=False)
config = env.configuration
config

{'episodeSteps': 1000,
 'actTimeout': 2,
 'runTimeout': 1200,
 'columns': 7,
 'rows': 6,
 'inarow': 4,
 'agentTimeout': 60,
 'timeout': 2}

In [12]:
def count_ngrams(window, inarow, mark):
    current = 0
    count = 0
    for i in range(len(window)):
        if window[current] == mark:
            current += 1
            if current == inarow:
                current -= 1
                count+=1
        else:
            current = 0
    return count
def expand_states(states, inarow, marks):
    # States is a numpy array [memories, rows, cols]
    # Transform it into a numpy array [memories, 4, rows, cols]
    expanded_states = np.zeros((states.shape[0], 2*(inarow-1)-1, states.shape[1], states.shape[2]))
    for n in range(states.shape[0]):
        mark_n = marks[n]
        opponent_mark_n = 1 if mark_n==-1 else -1
        for k in range(2*(inarow-1)-1):
            if 0==k:
                expanded_states[n,k,:,:] = states[n,:,:]
                continue
            else:
                inarow_k = int((k+1)/2)+1
                mark_k = mark_n if k%2==1 else opponent_mark_n
            for i in range(states.shape[1]):
                i_min = max(i-inarow_k+1, 0)
                i_max = min(i+inarow_k-1, states.shape[1]-1)
                for j in range(states.shape[2]):
                    j_min = max(j-inarow_k+1, 0)
                    j_max = min(j+inarow_k-1, states.shape[2]-1)
                    value = 0
                    value += count_ngrams(list(states[n, i_min:i_max, j]), inarow_k, mark_k)
                    value += count_ngrams(list(states[n, i, j_min:j_max]), inarow_k, mark_k)
                    value += count_ngrams(list(states[n, i_min:i_max, j_min:j_max].diagonal()), inarow_k, mark_k)
                    value += count_ngrams(list(np.flipud(states[n, i_min:i_max, j_min:j_max]).diagonal()), inarow_k, mark_k)
                    expanded_states[n][k][i][j] = value
    return expanded_states

In [13]:
class DQNA:
    def __init__(self, rows, columns, action_size, inarow):
        self.rows = rows
        self.columns = columns
        self.action_size = action_size
        self.memory = []
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.999954
        self.epsilon_min = 0.01
        self.learning_rate = 0.001
        self.tau = 0.005
        self.inarow = inarow
        self.steps_done = 0

        self.policy_net = nn.Sequential(
            nn.Conv2d(in_channels = 5, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = inarow, padding='same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels=128, out_channels = 256, kernel_size=inarow, padding='same'),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(in_features=256*(rows)*(columns), out_features=1024),
            nn.ELU(),
            nn.Linear(in_features=1024, out_features=128),
            nn.ELU(),
            nn.Linear(in_features=128, out_features=32),
            nn.ELU(),
            nn.Linear(in_features=32, out_features=action_size),
            nn.ELU()
        ).to(device)
        self.target_net = nn.Sequential(
            nn.Conv2d(in_channels = 5, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = inarow, padding='same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels=128, out_channels = 256, kernel_size=inarow, padding='same'),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(in_features=256*(rows)*(columns), out_features=1024),
            nn.ELU(),
            nn.Linear(in_features=1024, out_features=128),
            nn.ELU(),
            nn.Linear(in_features=128, out_features=32),
            nn.ELU(),
            nn.Linear(in_features=32, out_features=action_size),
            nn.ELU()
        ).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.learning_rate, amsgrad=True)

    def remember(self, state, action, reward, next_state, done, mark):
        self.memory.append((state, action, reward, next_state, done, mark))
        if len(self.memory) > 10000:
            self.memory.pop(0)
    def act(self, state, valid_actions, mark):
        if np.random.rand() <= self.epsilon:
            return random.choice(valid_actions)
        state = np.expand_dims(state, axis=(0))
        state = expand_states(state, self.inarow, np.array([mark]))
        state = torch.from_numpy(state.astype(np.float32)).to(device)
        with torch.no_grad():
            q_values = self.policy_net(state)[0]
        q_valid = [(a,q_values[a]) for a in valid_actions]
        return max(q_valid, key=lambda x:x[1])[0]
    def replay(self, batch_size=64):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)
        states = np.array([m[0] for m in minibatch], dtype=np.float32)
        actions = np.array([m[1] for m in minibatch], dtype=np.float32)
        rewards = np.array([[m[2]]*self.action_size for m in minibatch], dtype=np.float32)
        next_states = np.array([m[3] for m in minibatch], dtype=np.float32)
        dones = np.array([m[4] for m in minibatch])
        marks = np.array([m[5] for m in minibatch])
        states = expand_states(states, self.inarow, marks)
        next_states = expand_states(next_states, self.inarow, marks)

        non_final_mask = torch.tensor(tuple(map(lambda done: done is not True,
                                        dones)), device=device, dtype=torch.bool)
        non_final_next_states = torch.cat([torch.from_numpy(np.expand_dims(s,axis=(0)).astype(np.float32)) for s,done in zip(next_states,dones)
                                        if done is not True])     
        state_action_values = self.policy_net(torch.from_numpy(states.astype(np.float32)).to(device))
        next_state_values = torch.zeros_like(state_action_values, device=device)
        with torch.no_grad():
            next_state_values[non_final_mask] = self.target_net(non_final_next_states.to(device))
        expected_state_action_values = (next_state_values * self.gamma) + torch.from_numpy(rewards).to(device)
        loss = self.criterion(state_action_values.unsqueeze(1), expected_state_action_values.unsqueeze(1))
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_value_(self.policy_net.parameters(), 100)
        self.optimizer.step()
    def update_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon = self.epsilon * self.epsilon_decay
    def clone_to_target(self):
        target_net_state_dict = self.target_net.state_dict()
        policy_net_state_dict = self.policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*self.tau + target_net_state_dict[key]*(1-self.tau)
        self.target_net.load_state_dict(target_net_state_dict)

In [14]:
def get_valid_moves(board, config):
    return [c for c in range(config.columns) if board[c]==0]
def state_to_np(board,config):
    state = np.array(board).reshape(config.rows, config.columns).astype(np.float32)
    state[state == 2] = -1
    return state
def check_window(window,num_discs, piece, config):
    return (window.count(piece) == num_discs and window.count(0) == config.inarow - num_discs)
def count_windows(grid, num_discs, piece, config):
    num_windows = 0
    grid_2d = np.array(grid).reshape(config.rows, config.columns)
    for r in range(config.rows):
        for c in range(config.columns - config.inarow + 1):
            window = list(grid_2d[r,c:c+config.inarow])
            if check_window(window,num_discs,piece,config): num_windows += 1
    for r in range(config.rows - config.inarow + 1):
        for c in range(config.columns):
            window = list(grid_2d[r:r+config.inarow,c])
            if check_window(window,num_discs,piece,config): num_windows += 1
    for r in range(config.rows - config.inarow + 1):
        for c in range(config.columns - config.inarow + 1):
            window = list(grid_2d[r:r+config.inarow,c:c+config.inarow].diagonal())
            if check_window(window,num_discs,piece,config): num_windows += 1
            window = list(np.flipud(grid_2d[r:r+config.inarow,c:c+config.inarow]).diagonal())
            if check_window(window,num_discs,piece,config): num_windows += 1
    return num_windows
def get_shaped_reward(board,action,step_reward,config,done,mark):
    if done:
        if step_reward == 1:
            return 1.0
        elif step_reward == -1:
            return -1.0
        else:
            return 0.0
    opponent_mark = 1 if mark == -1 else -1
    my_threes = count_windows(board,3,mark,config)
    opp_threes = count_windows(board,3,opponent_mark,config)
    heuristic_score = (my_threes * 0.1) - (opp_threes * 0.2)
    if action == config.columns // 2:
        heuristic_score+=0.05
    return max(-0.8,min(0.8, heuristic_score))

In [15]:
def agent_function(observation, configuration):

    import os

    import io
    import numpy as np
    import random
    import base64
    import torch
    import torch.nn as nn

    def build_model(inarow, rows, columns, action_size):
        return nn.Sequential(
            nn.Conv2d(in_channels = 5, out_channels = 64, kernel_size = inarow, padding = 'same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels = 64, out_channels = 128, kernel_size = inarow, padding='same'),
            nn.ELU(),
            nn.Dropout(),
            nn.Conv2d(in_channels=128, out_channels = 256, kernel_size=inarow, padding='same'),
            nn.ELU(),
            nn.Flatten(),
            nn.Linear(in_features=256*(rows)*(columns), out_features=1024),
            nn.ELU(),
            nn.Linear(in_features=1024, out_features=128),
            nn.ELU(),
            nn.Linear(in_features=128, out_features=32),
            nn.ELU(),
            nn.Linear(in_features=32, out_features=action_size),
            nn.ELU()
        )

    model = build_model(
        rows=configuration.rows,
        columns=configuration.columns,
        action_size=configuration.columns,
        inarow=configuration.inarow
    )
    encoded_weights = """
    BASE64_PARAMS"""

    decoded = base64.b64decode(encoded_weights)
    buffer = io.BytesIO(decoded)
    model.load_state_dict(torch.load(buffer))


    def check_win(board, mark, config):
        columns = config.columns
        rows = config.rows

        def count(offset_row, offset_col):
            for r in range(rows):
                for c in range(columns):
                    if r + 3 * offset_row >= rows or r + 3 * offset_row < 0: continue
                    if c + 3 * offset_col >= columns or c + 3 * offset_col < 0: continue
                    if board[r * columns + c] == mark and board[(r + offset_row) * columns + (c + offset_col)] == mark and board[(r + 2 * offset_row) * columns + (c + 2 * offset_col)] == mark and board[(r + 3 * offset_row) * columns + (c + 3 * offset_col)] == mark:
                        return True
            return False

        return count(1, 0) or count(0, 1) or count(1, 1) or count(1, -1)

    def simulate_drop(board, col, mark, config):
        next_board = list(board)
        for row in range(config.rows-1, -1, -1):
            if next_board[row * config.columns + col] == 0:
                next_board[row * config.columns + col] = mark
                break
        return next_board

    board = observation.board
    mark = 1 if observation.mark == 1 else -1
    opponent = 1 if mark == -1 else -1

    valid_moves = [c for c in range(configuration.columns) if board[c] == 0]
    if len(valid_moves) == 0: return 0

    for col in valid_moves:
        next_board = simulate_drop(board, col, mark, configuration)
        if check_win(next_board, mark, configuration):
            return col

    for col in valid_moves:
        next_board = simulate_drop(board, col, opponent, configuration)
        if check_win(next_board, opponent, configuration):
            return col

    state = np.array(board).reshape(configuration.rows, configuration.columns).astype(np.float32)
    state[state == 2] = -1

    def count_ngrams(window, inarow, mark):
        current = 0
        count = 0
        for i in range(len(window)):
            if window[current] == mark:
                current += 1
                if current == inarow:
                    current -= 1
                    count+=1
            else:
                current = 0
        return count
    def expand_states(states, inarow, marks):
        # States is a numpy array [memories, rows, cols]
        # Transform it into a numpy array [memories, 4, rows, cols]
        expanded_states = np.zeros((states.shape[0], 2*(inarow-1)-1, states.shape[1], states.shape[2]))
        for n in range(states.shape[0]):
            mark_n = marks[n]
            opponent_mark_n = 1 if mark_n==-1 else -1
            for k in range(2*(inarow-1)-1):
                if 0==k:
                    expanded_states[n,k,:,:] = states[n,:,:]
                    continue
                else:
                    inarow_k = int((k+1)/2)+1
                    mark_k = mark_n if k%2==1 else opponent_mark_n
                for i in range(states.shape[1]):
                    i_min = max(i-inarow_k+1, 0)
                    i_max = min(i+inarow_k-1, states.shape[1]-1)
                    for j in range(states.shape[2]):
                        j_min = max(j-inarow_k+1, 0)
                        j_max = min(j+inarow_k-1, states.shape[2]-1)
                        value = 0
                        value += count_ngrams(list(states[n, i_min:i_max, j]), inarow_k, mark_k)
                        value += count_ngrams(list(states[n, i, j_min:j_max]), inarow_k, mark_k)
                        value += count_ngrams(list(states[n, i_min:i_max, j_min:j_max].diagonal()), inarow_k, mark_k)
                        value += count_ngrams(list(np.flipud(states[n, i_min:i_max, j_min:j_max]).diagonal()), inarow_k, mark_k)
                        expanded_states[n][k][i][j] = value
        return expanded_states
    
    state = expand_states(np.expand_dims(state, axis=(0)), configuration.inarow, np.array([mark]))
    
    with torch.no_grad():
        q_values = model(torch.from_numpy(state.astype(np.float32)))[0]

    q_valid = [(c, q_values[c]) for c in valid_moves]
    best_action = max(q_valid, key=lambda x: x[1])[0]

    return int(best_action)
def write_agent_to_file(function, file):
    with open(file, "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

In [16]:
if torch.cuda.is_available() or torch.backends.mps.is_available():
    num_episodes = [500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
else:
    num_episodes = [250, 250, 250, 250, 250, 250, 250, 250, 250, 250]
dqn_agent = DQNA(
    rows=config.rows,
    columns=config.columns,
    action_size=config.columns,
    inarow=config.inarow
)
print(f'Starting training...\nEpisode 0/{num_episodes}')
total_reward = 0
for ep in range(num_episodes[0]):
    if random.randint(0,1):
        trainer = env.train([None, "random"])
    else:
        trainer = env.train(['random', None])
    obs = trainer.reset()
    board = obs['board']
    state = state_to_np(board,config)
    done = False
    while not done:
        mark = 1 if obs['mark']==1 else -1
        valid_moves = get_valid_moves(board,config)
        action = dqn_agent.act(state,valid_moves,mark)
        dqn_agent.update_epsilon()
        obs,step_reward,done,info = trainer.step(int(action))
        next_board = obs['board']
        mark = 1 if obs['mark']==1 else -1
        current_reward = get_shaped_reward(next_board, action, step_reward, config, done, mark)
        next_state = state_to_np(next_board,config)
        dqn_agent.remember(state,action,current_reward,next_state,done,mark)
        state = next_state
        board = next_board
        total_reward += current_reward
        dqn_agent.replay()
        dqn_agent.clone_to_target()
    if (ep+1) % 50 == 0:
        print(f"Episode {ep+1}/{num_episodes[0]}, Total Reward = {total_reward:.2f}, Epsilon = {dqn_agent.epsilon:.2f}")
        total_reward = 0

for i in range(len(num_episodes) - 1):
    PATH = 'connectX.pt'
    torch.save(dqn_agent.policy_net.state_dict(), 'connectX.pt')
    print("Model saved")
    OUTPUT_PATH = 'agent_' + str(i+1) + '.py'
    no_params_path = 'submission_without_parameters.py'

    write_agent_to_file(agent_function, no_params_path)

    with open(PATH, 'rb') as f:
        raw_bytes = f.read()
        encoded_weights = base64.encodebytes(raw_bytes).decode()

    with open(no_params_path, 'r') as file:
        data = file.read()

    data = data.replace('BASE64_PARAMS', encoded_weights)

    with open(OUTPUT_PATH, 'w') as f:
        f.write(data)
        print('written agent with net parameters to', OUTPUT_PATH)

    print(f"{OUTPUT_PATH} successfully created!")

    out = sys.stdout
    submission = utils.read_file(OUTPUT_PATH)
    saved_agent = agent.get_last_callable(submission)
    sys.stdout = out


    for ep in range(num_episodes[i+1]):
        if random.randint(0,1):
            trainer = env.train([None, saved_agent])
        else:
            trainer = env.train([saved_agent, None])
        obs = trainer.reset()
        board = obs['board']
        state = state_to_np(board,config)
        done = False
        while not done:
            valid_moves = get_valid_moves(board,config)
            mark = 1 if obs['mark']==1 else -1
            action = dqn_agent.act(state,valid_moves, mark)
            dqn_agent.update_epsilon()
            obs,step_reward,done,info = trainer.step(int(action))
            next_board = obs['board']
            mark = 1 if obs['mark']==1 else -1
            current_reward = get_shaped_reward(next_board, action, step_reward, config, done, mark)
            next_state = state_to_np(next_board,config)
            dqn_agent.remember(state,action,current_reward,next_state,done, mark)
            state = next_state
            board = next_board
            total_reward += current_reward
            dqn_agent.replay()
            dqn_agent.clone_to_target()
        if (ep+1) % 50 == 0:
            print(f"Episode {ep+1}/{num_episodes[i+1]}, Total Reward = {total_reward:.2f}, Epsilon = {dqn_agent.epsilon:.2f}")
            total_reward = 0

Starting training...
Episode 0/[500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
Episode 50/500, Total Reward = -23.85, Epsilon = 0.98
Episode 100/500, Total Reward = -51.40, Epsilon = 0.95
Episode 150/500, Total Reward = -19.85, Epsilon = 0.93
Episode 200/500, Total Reward = -0.75, Epsilon = 0.91
Episode 250/500, Total Reward = -1.85, Epsilon = 0.89
Episode 300/500, Total Reward = -31.05, Epsilon = 0.86
Episode 350/500, Total Reward = -17.90, Epsilon = 0.84
Episode 400/500, Total Reward = -24.50, Epsilon = 0.82
Episode 450/500, Total Reward = -13.45, Epsilon = 0.80
Episode 500/500, Total Reward = -36.70, Epsilon = 0.79
Model saved
<function agent_function at 0x7fd709577d80> written to submission_without_parameters.py
written agent with net parameters to agent_1.py
agent_1.py successfully created!
Episode 50/500, Total Reward = -35.80, Epsilon = 0.77
Episode 100/500, Total Reward = -25.50, Epsilon = 0.75
Episode 150/500, Total Reward =

In [20]:
PATH = 'connectX.pt'
torch.save(dqn_agent.policy_net.to('cpu').state_dict(), 'connectX.pt')
print("Model saved")

Model saved


In [21]:
OUTPUT_PATH = 'submission.py'
no_params_path = 'submission_without_parameters.py'

write_agent_to_file(agent_function, no_params_path)

with open(PATH, 'rb') as f:
    raw_bytes = f.read()
    encoded_weights = base64.encodebytes(raw_bytes).decode()

with open(no_params_path, 'r') as file:
    data = file.read()

data = data.replace('BASE64_PARAMS', encoded_weights)

with open(OUTPUT_PATH, 'w') as f:
    f.write(data)
    print('written agent with net parameters to', OUTPUT_PATH)

print(f"{OUTPUT_PATH} successfully created!")

<function agent_function at 0x7fd709577d80> written to submission_without_parameters.py
written agent with net parameters to submission.py
submission.py successfully created!


In [22]:
import sys
from kaggle_environments import evaluate, make, utils, agent
out = sys.stdout
submission = utils.read_file('submission.py')
saved_agent = agent.get_last_callable(submission)
sys.stdout = out

env = make("connectx", debug=True)
env.run([saved_agent, saved_agent])
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")

Success!


In [60]:
import sys
from kaggle_environments import evaluate, make, utils, agent
out = sys.stdout
agent_path_1 = 'agent_19.py'
agent_1_file = utils.read_file(agent_path_1)
agent_1 = agent.get_last_callable(agent_1_file)
agent_path_2 = 'submission.py'
agent_2_file = utils.read_file(agent_path_2)
agent_2 = agent.get_last_callable(agent_2_file)
sys.stdout = out

env = make("connectx", debug=True)
winning_games = [0,0]
test_episodes = 5
for i in range(test_episodes):
    env.run([agent_2, agent_1])
    if (env.state[0].reward == 1) and (env.state[1].reward == -1):
        winning_games[0] = winning_games[0] + 1
    elif (env.state[0].reward == -1) and (env.state[1].reward == 1):
        winning_games[1] = winning_games[1] + 1
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")
print(winning_games)

Success!
[0, 5]
